In [1]:
import pandas as pd
import sqlite3
import random
import os
import hashlib
import re
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from config import FieldsEnum, DBNames, DBTables

In [2]:
def generate_barcode(nome, espansione, condizione):
    def clean(s):
        return re.sub(r"[^A-Z0-9]", "", s.upper())

    # parte leggibile
    base = f"{clean(nome)[:4]}-{clean(espansione)[:3]}-{clean(condizione)[:2]}"

    # hash deterministico
    raw = f"{nome}|{espansione}|{condizione}".upper()
    short_hash = hashlib.md5(raw.encode()).hexdigest()[:6].upper()

    return f"{base}-{short_hash}"

In [3]:
df_database = pd.read_csv(r"C:\Users\s.galati\Documents\export-Pokemon-27-05-2026.csv")
df_database.rename(columns={"expansion": FieldsEnum.Espansione.value, "expansionCode": FieldsEnum.Espansione_ID.value}, inplace=True)
df_stock = pd.read_csv(r"C:\Users\s.galati\Documents\inventory-report-Pokemon.csv")
df_stock[FieldsEnum.Prezzo_Acquisto.value] = ["0.01"] * len(df_stock)
df_stock["barcode"] = df_stock.apply(lambda row: generate_barcode(row[FieldsEnum.Nome.value], row[FieldsEnum.Espansione_ID.value], row[FieldsEnum.Condizione.value]), axis=1)
print(len(df_stock), len(df_database))

4924 82102


In [ ]:
df_database.columns

In [ ]:
df_stock.iloc[18]

#### Create Stock Table

In [ ]:
conn = sqlite3.connect(os.path.join("..", DBNames.MAIN_DB.value))
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {DBTables.STOCK.value}")
query = f"""CREATE TABLE IF NOT EXISTS {DBTables.STOCK.value} (
    {FieldsEnum.ID_Cardmarket.value} INTEGER PRIMARY KEY,"""
for col in df_stock.columns:
    if col != FieldsEnum.ID_Cardmarket.value:
        query += f"\n    '{col}' TEXT,"
query = query.rstrip(",") + "\n)"
cursor.execute(query)


# Scrive la tabella nel database
df_stock.to_sql(
    DBTables.STOCK.value,
    conn,
    if_exists="append",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)
conn.close()

### Create Main Database

#### Create unpriced table

In [ ]:
conn = sqlite3.connect(os.path.join("..", DBNames.MAIN_DB.value))
cursor = conn.cursor()

cursor.execute(f"DROP TABLE IF EXISTS {DBTables.UNPRICED_CARDS.value}")
cursor.execute(f"""
CREATE TABLE "{DBTables.UNPRICED_CARDS.value}" (
    "{FieldsEnum.ID_Carta.value}" TEXT,
    "{FieldsEnum.Nome.value}" TEXT,  
    "{FieldsEnum.Espansione_ID.value}" TEXT,
    "{FieldsEnum.Espansione.value}" TEXT,  
    "{FieldsEnum.Condizione.value}" TEXT,      
    "{FieldsEnum.Barcode.value}" TEXT,
    "{FieldsEnum.Prezzo.value}" REAL,
    "{FieldsEnum.Quantità.value}" INTEGER,
    "{FieldsEnum.Prezzo_acquisto.value}" REAL,
    "{FieldsEnum.Da_Prezzare.value}" TEXT
)
""")

conn.commit()
conn.close()

#### Create sales table

In [ ]:
conn = sqlite3.connect(os.path.join("..", DBNames.MAIN.value))
cursor = conn.cursor()

cursor.execute(f"DROP TABLE IF EXISTS {DBTables.SALES.value}")

cursor.execute(f"""
CREATE TABLE "{DBTables.SALES.value}" (
    "{FieldsEnum.ID_Vendita.value}" INTEGER PRIMARY KEY AUTOINCREMENT,
    "{FieldsEnum.Barcode.value}" TEXT,
    "{FieldsEnum.Espansione.value}" TEXT,
    "{FieldsEnum.Espansione_ID.value}" TEXT,
    "{FieldsEnum.Nome.value}" TEXT,
    "{FieldsEnum.Condizione.value}" TEXT,
    "{FieldsEnum.Prezzo.value}" REAL,
    "{FieldsEnum.Prezzo_Vendita.value}" REAL,
    "{FieldsEnum.Data_Vendita.value}" TEXT
)
""")

conn.commit()
conn.close()

#### Create purchase table

In [ ]:
conn = sqlite3.connect(os.path.join("..", DBNames.MAIN.value))
cursor = conn.cursor()

cursor.execute(f"DROP TABLE IF EXISTS {DBTables.PURCHASES.value}")

cursor.execute(f"""
CREATE TABLE "{DBTables.PURCHASES.value}" (
    "{FieldsEnum.ID_Acquisto.value}" INTEGER PRIMARY KEY AUTOINCREMENT,
    "{FieldsEnum.Barcode.value}" TEXT,
    "{FieldsEnum.Espansione_ID.value}" TEXT,
    "{FieldsEnum.Espansione.value}" TEXT,
    "{FieldsEnum.Nome.value}" TEXT,
    "{FieldsEnum.Condizione.value}" TEXT,
    "{FieldsEnum.Prezzo.value}" REAL,
    "{FieldsEnum.Data_Acquisto.value}" TEXT
)
""")

conn.commit()
conn.close()

#### Create draft db

In [2]:
conn = sqlite3.connect(os.path.join("..", DBNames.MAIN_DB.value))
cursor = conn.cursor()

cursor.execute(f"DROP TABLE IF EXISTS {DBTables.BOZZE_ACQUISTI.value}")

cursor.execute(f"""
CREATE TABLE "{DBTables.BOZZE_ACQUISTI.value}" (
    "{FieldsEnum.ID_Bozza_Acquisto.value}" INTEGER PRIMARY KEY AUTOINCREMENT,
    "{FieldsEnum.Nome.value}" TEXT,
    "{FieldsEnum.Numero_oggetti.value}" TEXT,
    "{FieldsEnum.Totale.value}" TEXT,
    "{FieldsEnum.Oggetti.value}" TEXT
)
""")

conn.commit()
conn.close()

#### CREATE INVENTARY DB

In [10]:
conn = sqlite3.connect(os.path.join("..", DBNames.CARD_DB.value))
cursor = conn.cursor()

In [11]:
df_database.to_sql(
    DBTables.DATABASE_CARDS.value,
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)
conn.close()

In [19]:
df_database.columns

Index(['cardmarketId', 'name', 'collectorNumber', 'rarity', 'expansion',
       'expansionCode', 'scryfallId', 'tcgplayerId'],
      dtype='object')